# Ridge Regression Practice

**Topic:** Linear Regression &nbsp;·&nbsp; **Difficulty:** Medium

## The data (created in the next cell)

- `X_train` `(120, 60)`, `y_train` `(120,)`: only 8 features truly matter. The remaining 52 are correlated distractors.
- `X_test` `(400, 60)`: predict these. `y_test` is used by the evaluation cell.
- Try plain `LinearRegression` first and note its test R², then regularize.

## Your task

1. Fit an L2-regularized linear model (`Ridge` / `RidgeCV`, or your own normal equations with a `λI` term).
2. Choose the regularization strength with cross-validation, not by eyeballing.
3. Set `predictions` to your predictions on `X_test`.

## Evaluation

Test R² must reach the target shown in the evaluation cell. Only the test R² is graded. The cross-validation requirement is on your honour.


In [ ]:
import numpy as np

rng = np.random.default_rng(23)
n_train, n_test, d, d_true = 120, 400, 60, 8

basis = rng.normal(size=(d_true + 10, d))     # low-dim structure -> correlated features
w_true = np.zeros(d); w_true[:d_true] = rng.normal(0, 3.0, d_true)

def make(n, seed):
    r = np.random.default_rng(seed)
    Z = r.normal(size=(n, d_true + 10))
    X = Z @ basis + 0.3 * r.normal(size=(n, d))
    y = X @ w_true + r.normal(0, 9.0, n)
    return X, y

X_train, y_train = make(n_train, 31)
X_test, y_test = make(n_test, 32)
print(f"X_train {X_train.shape}  (correlated features: regularize!)")


In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.





In [ ]:
# =====================================================================
# EVALUATION: run this cell to compare your predictions to the truth
# =====================================================================
import numpy as np

TARGET_R2 = 0.75

if "predictions" not in dir():
    raise NameError("Define `predictions` in the solution cell before running this cell.")

_pred = np.asarray(predictions, dtype=float).ravel()
_true = np.asarray(y_test, dtype=float).ravel()
if _pred.shape != _true.shape:
    raise ValueError(f"`predictions` has shape {_pred.shape}, expected {_true.shape}.")

_rmse = float(np.sqrt(np.mean((_pred - _true) ** 2)))
_ss_res = float(np.sum((_true - _pred) ** 2))
_ss_tot = float(np.sum((_true - _true.mean()) ** 2))
r2 = 1.0 - _ss_res / _ss_tot
print("=" * 52)
print(f"Test RMSE : {_rmse:.4f}")
print(f"Test R^2  : {r2:.4f}   (target: >= {TARGET_R2:.2f})")
print(f"Worst absolute error: {float(np.max(np.abs(_pred - _true))):.4f}")
print("=" * 52)
if r2 >= TARGET_R2:
    print("PASSED, nice work!")
else:
    print("NOT PASSED yet: revisit your model and try again.")


## Solution

With 120 samples and 60 correlated features, plain least squares overfits badly. `RidgeCV` sweeps the L2 strength over a log grid with cross-validation and picks the value that generalizes.

```python
import numpy as np
from sklearn.linear_model import RidgeCV

model = RidgeCV(alphas=np.logspace(-2, 4, 25), cv=5)
model.fit(X_train, y_train)
print(f"chosen alpha: {model.alpha_:.4g}")
predictions = model.predict(X_test)
```
